# BIS Policy Rate Monitor — Data Exploration

This notebook explores the raw BIS **Central bank policy rates** bulk-download file before transformation.

In [21]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

## 1. Load the raw BIS bulk download

In [22]:
DATA_PATH = Path("../data/raw/WS_CBPOL_csv_flat.zip")

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Raw BIS dataset not found: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(
    DATA_PATH,
    compression="zip",
    low_memory=False,
)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

Rows: 731,976
Columns: 18


In [23]:
df.sample(5, random_state=42)

,STRUCTURE,STRUCTURE_ID,ACTION,FREQ:Frequency,REF_AREA:Reference area,TIME_PERIOD:Time period or range,OBS_VALUE:Observation Value,UNIT_MEASURE:Unit of measure,UNIT_MULT:Unit Multiplier,TIME_FORMAT:Time Format,COMPILATION:Compilation,DECIMALS:Decimals,SOURCE_REF:Publication Source,SUPP_INFO_BREAKS:Supplemental information and breaks,TITLE:Title,OBS_STATUS:Observation Status,OBS_CONF:Observation confidentiality,OBS_PRE_BREAK:Pre-Break Observation
698358,dataflow,BIS:WS_CBPOL(1.0): Central bank policy rates,I,D: Daily,BR: Brazil,2025-08-29,15.0,368: Per cent per year,0: Units,NaN,"From 5 Mar 1999 onwards Central Bank target, money market (SELIC) overnight rate; from 1 Jul 1996 to 4 Mar 1999: Cen...",4: Four,Central Bank of Brazil,"This rate can be considered the official policy rate as from 5 Mar 1999. From Jul 1996 to 4 Mar 1999, the TBC rate (...",Central bank policy rates - Brazil - Daily - End of period,A: Normal value,F: Free,NaN
669835,dataflow,BIS:WS_CBPOL(1.0): Central bank policy rates,I,D: Daily,AT: Austria,1964-06-12,4.5,368: Per cent per year,0: Units,NaN,From 1 Jan 1999 onwards: the series is discontinued as Austria joined the euro area; from 2 Sep 1996 to 31 Dec 1998:...,4: Four,"Central Bank of the Republic of Austria, Bank for International Settlements",NaN,Central bank policy rates - Austria - Daily - End of period,A: Normal value,F: Free,NaN
359358,dataflow,BIS:WS_CBPOL(1.0): Central bank policy rates,I,D: Daily,KR: Korea,2018-08-13,1.5,368: Per cent per year,0: Units,NaN,From 7 Mar 2008 onwards: Bank of Korea base rate; from 6 May 1999 to 6 Mar 2008: target overnight call rate(base rat...,4: Four,Bank of Korea,The Bank of Korea base rate has been changed from the overnight call rate to 7 day repurchase agreements rate since ...,Central bank policy rates - Korea - Daily - End of period,A: Normal value,F: Free,NaN
102177,dataflow,BIS:WS_CBPOL(1.0): Central bank policy rates,I,D: Daily,DE: Germany,1958-05-13,3.5,368: Per cent per year,0: Units,NaN,From 1 Jan 1999 onwards: the series is discontinued as Germany joined the euro area; from 1 Jan 1985 to 31 Dec 1998:...,4: Four,"Deutsche Bundesbank, Bank for International Settlements","Until the end of 1998, the refinancing policy of the Deutsche Bundesbank was essentially based on discount and margi...",Central bank policy rates - Germany - Daily - End of period,A: Normal value,F: Free,NaN
724603,dataflow,BIS:WS_CBPOL(1.0): Central bank policy rates,I,D: Daily,AR: Argentina,2005-05-01,NaN,368: Per cent per year,0: Units,NaN,From 10 July 2025 onwards: no policy rate adopted; from 22 July 2024 to 9 July 2025: liquidity absorption rate for t...,4: Four,Central Bank of Argentina,"From 10 July 2025 onwards, the monetary policy of the Central Bank of Argentina is based on controlling monetary agg...",Central bank policy rates - Argentina - Daily - End of period,M: Missing value; data cannot exist,F: Free,NaN


## 2. Inspect the raw schema

Before writing transformation code, inspect the exact column names and data types delivered by BIS.

In [24]:
pd.DataFrame(
    {
        "column": df.columns,
        "dtype": df.dtypes.astype(str).values,
        "non_null": df.notna().sum().values,
        "missing": df.isna().sum().values,
        "missing_pct": (df.isna().mean().values * 100).round(2),
    }
)

,column,dtype,non_null,missing,missing_pct
0,STRUCTURE,str,731976,0,0.00
1,STRUCTURE_ID,str,731976,0,0.00
2,ACTION,str,731976,0,0.00
3,FREQ:Frequency,str,731976,0,0.00
4,REF_AREA:Reference area,str,731976,0,0.00
5,TIME_PERIOD:Time period or range,str,731976,0,0.00
6,OBS_VALUE:Observation Value,float64,662370,69606,9.51
7,UNIT_MEASURE:Unit of measure,str,731976,0,0.00
8,UNIT_MULT:Unit Multiplier,str,731976,0,0.00
9,TIME_FORMAT:Time Format,float64,0,731976,100.00


In [25]:
df.columns.tolist()

['STRUCTURE',
 'STRUCTURE_ID',
 'ACTION',
 'FREQ:Frequency',
 'REF_AREA:Reference area',
 'TIME_PERIOD:Time period or range',
 'OBS_VALUE:Observation Value',
 'UNIT_MEASURE:Unit of measure',
 'UNIT_MULT:Unit Multiplier',
 'TIME_FORMAT:Time Format',
 'COMPILATION:Compilation',
 'DECIMALS:Decimals',
 'SOURCE_REF:Publication Source',
 'SUPP_INFO_BREAKS:Supplemental information and breaks',
 'TITLE:Title',
 'OBS_STATUS:Observation Status',
 'OBS_CONF:Observation confidentiality',
 'OBS_PRE_BREAK:Pre-Break Observation']

## 3. Inspect important BIS dimensions and attributes

The flat SDMX export contains both observation fields and metadata attached to the series.

These fields are especially relevant to the assignment because the generated outputs should retain useful attributes such as the series title, units, multiplier, decimals, and compilation information.

In [26]:
FIELDS_TO_INSPECT = [
    "FREQ",
    "REF_AREA",
    "UNIT_MEASURE",
    "UNIT_MULT",
    "DECIMALS",
    "OBS_STATUS",
    "OBS_CONF",
    "OBS_PRE_BREAK",
]

for column in FIELDS_TO_INSPECT:
    if column in df.columns:
        print(f"\n{column}")
        print(df[column].value_counts(dropna=False).head(15))

## 4. Countries and reporting frequencies


In [27]:
if "REF_AREA" in df.columns:
    areas = (
        df["REF_AREA"]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .sort_values()
    )

    print(f"Unique reference areas: {len(areas)}")
    display(areas.head(30).to_frame("REF_AREA"))

In [28]:
if {"REF_AREA", "FREQ"}.issubset(df.columns):
    frequency_by_area = (
        df.groupby(["REF_AREA", "FREQ"], dropna=False)
        .size()
        .rename("rows")
        .reset_index()
        .sort_values(["REF_AREA", "FREQ"])
    )

    frequency_by_area.head(30)

## 5. Create temporary exploration columns

In [29]:
explore = df.copy()

if "REF_AREA" in explore.columns:
    area = explore["REF_AREA"].astype("string").str.split(":", n=1, expand=True)
    explore["country_code"] = area[0].str.strip()
    explore["country_name"] = area[1].str.strip()

if "FREQ" in explore.columns:
    explore["frequency"] = (
        explore["FREQ"]
        .astype("string")
        .str.split(":", n=1)
        .str[0]
        .str.strip()
    )

if "OBS_VALUE" in explore.columns:
    explore["observation_value"] = pd.to_numeric(
        explore["OBS_VALUE"],
        errors="coerce",
    )

explore[
    [
        column
        for column in [
            "country_code",
            "country_name",
            "frequency",
            "TIME_PERIOD",
            "observation_value",
        ]
        if column in explore.columns
    ]
].head()

""
0
1
2
3
4


## 6. Check missing observations


In [30]:
if "OBS_VALUE" in df.columns:
    print(f"Missing OBS_VALUE: {df['OBS_VALUE'].isna().sum():,}")
    print(f"Share missing: {df['OBS_VALUE'].isna().mean():.2%}")

    df.loc[
        df["OBS_VALUE"].isna(),
        [
            column
            for column in [
                "REF_AREA",
                "FREQ",
                "TIME_PERIOD",
                "OBS_VALUE",
                "OBS_STATUS",
                "OBS_CONF",
            ]
            if column in df.columns
        ],
    ].head(20)

## 7. Check potential duplicate observation keys


In [31]:
duplicate_key = [
    column
    for column in ["country_code", "frequency", "TIME_PERIOD"]
    if column in explore.columns
]

if len(duplicate_key) == 3:
    duplicates = explore.loc[
        explore.duplicated(
            subset=duplicate_key,
            keep=False,
        )
    ].sort_values(duplicate_key)

    print(f"Rows involved in duplicate keys: {len(duplicates):,}")
    display(duplicates.head(20))

## 8. Explore the report countries


In [32]:
REPORT_COUNTRIES = ["US", "XM", "GB", "JP", "CH", "FR"]

if "country_code" in explore.columns:
    selected = explore.loc[
        explore["country_code"].isin(REPORT_COUNTRIES)
    ].copy()

    (
        selected.groupby(
            ["country_code", "country_name", "frequency"],
            dropna=False,
        )
        .size()
        .rename("rows")
        .reset_index()
        .sort_values(["country_code", "frequency"])
    )

## 9. Date coverage


In [33]:
def parse_exploration_date(frame: pd.DataFrame) -> pd.Series:
    dates = pd.Series(pd.NaT, index=frame.index, dtype="datetime64[ns]")

    daily = frame["frequency"].eq("D")
    monthly = frame["frequency"].eq("M")

    dates.loc[daily] = pd.to_datetime(
        frame.loc[daily, "TIME_PERIOD"],
        errors="coerce",
    )

    dates.loc[monthly] = pd.to_datetime(
        frame.loc[monthly, "TIME_PERIOD"],
        format="%Y-%m",
        errors="coerce",
    )

    return dates


if {"frequency", "TIME_PERIOD"}.issubset(explore.columns):
    explore["observation_date"] = parse_exploration_date(explore)

    print(
        "Unparsed dates:",
        explore["observation_date"].isna().sum(),
    )

In [42]:
if {"country_code", "observation_date"}.issubset(explore.columns):
    coverage = (
        explore.loc[
            explore["country_code"].isin(REPORT_COUNTRIES)
            & explore["observation_date"].notna()
        ]
        .groupby(["country_code", "frequency"])
        .agg(
            first_date=("observation_date", "min"),
            last_date=("observation_date", "max"),
            observations=("observation_date", "size"),
        )
        .reset_index()
    )

    print(coverage)

## 10. Latest available observations



In [35]:
if {
    "country_code",
    "observation_date",
    "observation_value",
}.issubset(explore.columns):
    latest_available = (
        explore.loc[
            explore["country_code"].isin(REPORT_COUNTRIES)
            & explore["observation_value"].notna()
            & explore["observation_date"].notna()
        ]
        .sort_values("observation_date")
        .groupby(["country_code", "frequency"], as_index=False)
        .tail(1)
    )

    latest_available[
        [
            "country_code",
            "country_name",
            "frequency",
            "observation_date",
            "observation_value",
        ]
    ].sort_values("country_code")

## 11. France as a discontinued-series example


In [36]:
if "country_code" in explore.columns:
    france = explore.loc[
        explore["country_code"].eq("FR")
        & explore["observation_value"].notna()
    ].sort_values("observation_date")

    if not france.empty:
        display(
            france[
                [
                    column
                    for column in [
                        "country_code",
                        "country_name",
                        "frequency",
                        "observation_date",
                        "observation_value",
                        "COMPILATION",
                    ]
                    if column in france.columns
                ]
            ].tail()
        )

        if "COMPILATION" in france.columns:
            print("\nLatest compilation note:\n")
            print(france.iloc[-1]["COMPILATION"])

## 12. Rate ranges since 2015


In [37]:
START_DATE = pd.Timestamp("2015-01-01")

if {
    "country_code",
    "observation_date",
    "observation_value",
}.issubset(explore.columns):
    since_start = explore.loc[
        explore["country_code"].isin(REPORT_COUNTRIES)
        & explore["observation_date"].ge(START_DATE)
        & explore["observation_value"].notna()
    ].copy()

    rate_ranges = (
        since_start.groupby(
            ["country_code", "country_name"],
            as_index=False,
        )
        .agg(
            observations=("observation_value", "size"),
            minimum_rate=("observation_value", "min"),
            maximum_rate=("observation_value", "max"),
            first_date=("observation_date", "min"),
            last_date=("observation_date", "max"),
        )
    )

    rate_ranges

## 13. Inspect series metadata


In [38]:
metadata_columns = [
    "REF_AREA",
    "FREQ",
    "TITLE",
    "UNIT_MEASURE",
    "UNIT_MULT",
    "DECIMALS",
    "COMPILATION",
    "SOURCE_REF",
    "SUPP_INFO_BREAKS",
]

existing_metadata = [
    column
    for column in metadata_columns
    if column in df.columns
]

if existing_metadata:
    metadata = (
        df.loc[
            explore["country_code"].isin(REPORT_COUNTRIES),
            existing_metadata,
        ]
        .drop_duplicates()
        .sort_values(
            [
                column
                for column in ["REF_AREA", "FREQ"]
                if column in existing_metadata
            ]
        )
    )

    metadata.head(20)

## 14. Plot selected policy-rate series


In [43]:
plot_data = explore.loc[
    explore["country_code"].isin(["US", "XM", "GB", "JP", "CH"])
    & explore["frequency"].eq("D")
    & explore["observation_date"].ge(START_DATE)
    & explore["observation_value"].notna()
].copy()

fig, ax = plt.subplots(figsize=(11, 6))

for country_code, country in plot_data.groupby("country_code"):
    country = country.sort_values("observation_date")

    ax.plot(
        country["observation_date"],
        country["observation_value"],
        label=country_code,
    )

ax.set_title("BIS Central Bank Policy Rates")
ax.set_xlabel("Date")
ax.set_ylabel("Policy rate")
ax.legend()
fig.tight_layout()

plt.show()

KeyError: 'country_code'